# Ultralytics detectors: YOLO11 (one-stage, anchor-free) and RT-DETR (query-based)

COCO-pretrained models fine-tuned on every 5th training frame. In training images, ignore regions are greyed out and boxes more than half inside them are dropped. Validation and test frames are predicted as they are; the evaluator skips detections in ignore regions (`iscrowd`). Each run is one entry in `EXPERIMENTS`, and `RUN` picks it.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import RTDETR, YOLO

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load, make_yolo_dataset, select_frames
from src.evaluate import evaluate, to_coco
from src.predict import predict_yolo
from src.submission import write_submission

KAGGLE = Path("/kaggle/input").exists()
DATA = next(Path("/kaggle/input").rglob("train/annotations_coco.json")).parents[1] if KAGGLE else ROOT / "data/raw"
WORK = Path("/kaggle/working") if KAGGLE else ROOT / "runs/local"
PREPARED = Path("/kaggle/temp/yolo") if KAGGLE else WORK / "yolo"
GPUS = torch.cuda.device_count()
DEVICE = list(range(GPUS)) if GPUS > 1 else 0 if GPUS else "mps"  # a list of GPUs means DDP

EXPERIMENTS = {  # batch is per GPU; 1280 px with batch 8 takes 13.6 of the T4's 15 GB
    "yolo11s_1280_20ep": dict(model="yolo11s.pt", imgsz=1280, epochs=20, batch=8, optimizer="SGD", lr0=0.01),
    "yolo11s_1280_5ep_lr001": dict(model="yolo11s.pt", imgsz=1280, epochs=5, batch=8, optimizer="SGD", lr0=0.001),
    "yolo11s_1536_5ep_lr001": dict(model="yolo11s.pt", imgsz=1536, epochs=5, batch=4, optimizer="SGD", lr0=0.001),
    "rtdetr-l_960_10ep": dict(model="rtdetr-l.pt", imgsz=960, epochs=10, batch=4, optimizer="AdamW", lr0=0.0001),
}
RUN = "yolo11s_1280_5ep_lr001"
CFG = {**EXPERIMENTS[RUN], "train_stride": 5, "curve_stride": 10, "seed": 42}
BATCH = CFG["batch"] * max(GPUS, 1)
Model = RTDETR if CFG["model"].startswith("rtdetr") else YOLO
RUN_DIR = next(Path("/kaggle/input").rglob(RUN), None) if KAGGLE else None  # finished run attached as input
print(RUN, "|", GPUS, "x", torch.cuda.get_device_name(0) if GPUS else DEVICE)

## Data

Training frames are resized once so the long side equals `imgsz`, which makes JPEG decoding in the data loader cheap.

In [ ]:
train_coco, val_coco = load(DATA / "train/annotations_coco.json"), load(DATA / "val/annotations_coco.json")
test = pd.read_csv(DATA / "test_images.csv")
t = time.time()
data_yaml = make_yolo_dataset(train_coco, val_coco, DATA, PREPARED, CFG["train_stride"], CFG["curve_stride"], CFG["imgsz"])
train_frames = len(select_frames(train_coco, CFG["train_stride"]))
del train_coco  # 1.1M train boxes are not needed after export
print(f"{train_frames} train frames prepared in {time.time() - t:.0f}s")

## Training

SGD for YOLO, AdamW for RT-DETR, cosine decay, AMP, mosaic switched off for the last quarter of the epochs. Ultralytics validates every epoch on every 10th validation frame (masked); that draws the training curve and picks `best.pt`. Reported metrics come from the full validation split below. If a finished run's output is attached as input, training is skipped.

In [ ]:
if RUN_DIR is None:
    Model(CFG["model"]).train(data=str(data_yaml), imgsz=CFG["imgsz"], epochs=CFG["epochs"], batch=BATCH, device=DEVICE,
                              seed=CFG["seed"], optimizer=CFG["optimizer"], lr0=CFG["lr0"], cos_lr=True,
                              close_mosaic=max(CFG["epochs"] // 4, 1),
                              cache="ram" if GPUS == 1 else False,  # each DDP process would hold its own copy
                              project=str(WORK), name=RUN, exist_ok=True, plots=False)
    RUN_DIR = WORK / RUN
print(RUN_DIR)

In [ ]:
curve = pd.read_csv(RUN_DIR / "results.csv")
train_hours = curve.time.iloc[-1] / 3600
plt.figure(figsize=(6, 3.5))
plt.plot(curve.epoch, curve["metrics/mAP50-95(B)"], marker="o", label="mAP@[.5:.95]")
plt.plot(curve.epoch, curve["metrics/mAP50(B)"], marker="o", label="AP50")
plt.xlabel("epoch")
plt.title(f"{RUN}: Ultralytics val, every {CFG['curve_stride']}th frame")
plt.legend()
plt.show()
print(f"training time: {train_hours:.2f} h")

## Validation: all 2,846 frames, best checkpoint

In [ ]:
model = Model(RUN_DIR / "weights/best.pt")
val_frames = [(i["id"], DATA / "val" / i["file_name"]) for i in select_frames(val_coco)]
val_dets, ms_img = predict_yolo(model, val_frames, CFG["imgsz"])
metrics = evaluate(val_coco, val_dets)
(WORK / "val_predictions.json").write_text(json.dumps(to_coco(val_dets)))
pd.DataFrame([{**metrics, "ms_img": ms_img, "empty_images": sum(len(b) == 0 for b, _, _ in val_dets.values())}]).round(4)

## Test predictions

In [ ]:
test_dets, _ = predict_yolo(model, [(r.image_id, DATA / r.file_name) for r in test.itertuples()], CFG["imgsz"])
write_submission(test, test_dets, WORK / "submission.csv")
row = dict(run_id=RUN, status="completed", model=CFG["model"], seed=CFG["seed"], train_stride=CFG["train_stride"],
           train_frames=train_frames, val_frames=len(val_frames), epochs=CFG["epochs"], imgsz=CFG["imgsz"], **metrics,
           ms_img=ms_img, gpu_hours=train_hours, public_lb=None,
           notes=f'{max(GPUS, 1)} GPU, batch {BATCH}, {CFG["optimizer"]} {CFG["lr0"]} cosine, best.pt')
pd.DataFrame([row]).to_csv(WORK / "experiment.csv", index=False)
print("DONE:", len(test), "rows in submission.csv")